In [10]:
import time

import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [6]:
kr = "\n".join(line for line in open("data/kr.txt", encoding="utf-8").read().splitlines() if line.strip())
eng = "\n".join(line for line in open("data/eng.txt", encoding="utf-8").read().splitlines() if line.strip())
chn = "\n".join(line for line in open("data/chn.txt", encoding="utf-8").read().splitlines() if line.strip())

kr_tokens = list(map(int, kr.encode('utf-8')))
en_tokens = list(map(int, eng.encode('utf-8')))
ch_tokens = list(map(int, chn.encode('utf-8')))
tokens = kr_tokens + en_tokens + ch_tokens
print(f"len(kr_tokens):{len(kr_tokens)}", kr_tokens[: 10])
print(f"len(en_tokens):{len(en_tokens)}", en_tokens[: 10])
print(f"len(ch_tokens):{len(ch_tokens)}", ch_tokens[: 10])
print(f"len(tokens):{len(tokens)}", list(tokens)[: 1000])


FileNotFoundError: [Errno 2] No such file or directory: '/home/jeffjin/projects/ai-ml/py-ai/zero-to-hero/mini-tokenizer'

In [ ]:
from collections import Counter, defaultdict

def get_ngram_stats(ids, min_n=2, max_n=5):
    if min_n < 2 or max_n < 2 or min_n >= max_n:
        raise ValueError("min_n and max_n must be greater than or equal to 2.")

    stats = defaultdict(int)
    size = len(ids)
    for n in range(min_n, max_n + 1):
        for i in range(size - n + 1):
            ngram = tuple(ids[i:i+n])
            stats[ngram] += 1
    return stats

def get_status(ids):
    stats = {}
    # for i in range(len(ids) - 1):
    #     a, b = ids[i], ids[i + 1]
    #     pair = (a, b)
    #     stats[pair] = stats.get(pair, 0) + 1
    for a, b in zip(ids, ids[1:]):
        pair = (a, b)
        stats[pair] = stats.get(pair, 0) + 1
    return stats

status = get_status(tokens)
sorted_status = sorted(((v, k) for (k, v) in status.items()), reverse=True)
print(f"len(sorted_status):{len(sorted_status)}", list(sorted_status)[: 10])
print("most common bigrams:", [chr(a) + chr(b) for (_, (a, b)) in sorted_status[: 10]])
chr(32) + chr(236)

In [ ]:
def get_stats(ids):
    return Counter(zip(ids, ids[1:]))

def merge_once(ids, pair, new_token):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_token)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

def train_bpe(ids, start_token=256, threshold=5):
    bpe_map = {}  # new_token -> (a, b)
    new_token = start_token
    ids = list(ids)

    while True:
        stats = get_stats(ids)
        if not stats:
            break
        pair, freq = stats.most_common(1)[0]
        if freq < threshold:   # merge while freq >= threshold
            break
        # print(f"merging pair {pair} with frequency {freq} into new token {new_token}")
        ids = merge_once(ids, pair, new_token)
        bpe_map[new_token] = pair
        new_token += 1

    return ids, bpe_map

new_ids, bpe_map = train_bpe(tokens, start_token=256, threshold=10)


In [ ]:
print(f"len(original_ids):{len(tokens)}, len(new_ids):{len(new_ids)}", new_ids[: 1000])
print(f"len(bpe_map):{len(bpe_map)}", list(sorted(bpe_map.items(), reverse=True))[: 1000])

In [ ]:
def expand_bytes(token_id, bpe_map):
    if token_id < 256:
        return bytes([token_id])
    a, b = bpe_map[token_id]
    return expand_bytes(a, bpe_map) + expand_bytes(b, bpe_map)

def decode(token_ids, bpe_map):
    data = b''.join(expand_bytes(t, bpe_map) for t in token_ids)
    return data.decode('utf-8', errors='replace')

decode(new_ids, bpe_map)

In [ ]:
def encode(text, bpe_map):
    tokens = list(map(int, text.encode('utf-8')))
    reversed_bpe_map = {v: k for k, v in bpe_map.items()}
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: reversed_bpe_map.get(p, float('inf')))
        if pair not in reversed_bpe_map:
            break
        idx = reversed_bpe_map[pair]
        tokens = merge_once(tokens, pair, idx)
    return tokens

print(encode("hello world!", bpe_map))

In [ ]:
print(decode(encode("AI는 여기로 연결됩니다. ", bpe_map), bpe_map))

In [ ]:
import regex as re

gpt2 = re.compile(r"""'s|'t|'re|'ve|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2, "Hello world how are you"))

In [ ]:
import tiktoken

# GPT-2 (does not merge spaces)
# enc = tiktoken.get_encoding("gpt2")
# print(enc.encode("hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode(" hello world!!!"))